In [ ]:
# Since some cells will fail on purpose in this notebook, only report minimal (meaningful) error messages
%xmode minimal

import os
# For a fair compiraison of performance against NumPy, we disable multithreading
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
os.environ["OMP_NUM_THREADS"] = "1"

import jax

# For a fair comparaison of performance against NumPy, we use doubles for JAX computations
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Introduction to JAX

JAX is a library for high-performance numerical and scientific computing.
Its array API deliberately mirrors NumPy's.
Most NumPy code ports over by swapping `import numpy as np` for `import jax.numpy as jnp`.
So if you know NumPy, you already know most of JAX's surface.

What sets JAX apart is a small set of [composable function transformations](https://docs.jax.dev/en/latest/key-concepts.html#transformations) that take an ordinary Python function and hand back a transformed one.
In this notebook we will explore the main one: `jax.jit`, that is, just-in-time compilation.
Other transformations exists and will be explored in further notebooks (such as `grad`, `vmap`, etc)

We'll take a first, hands-on look at JAX, building up from arrays and purity through `jit`
and its consequences, comparing against NumPy along the way to see both what carries over
and what changes.

For 80% of the time, JAX can be used like NumPy, like in this simple example:

In [ ]:
x = jnp.linspace(0., 2.*jnp.pi)
y = jnp.sin(x)

plt.plot(x, y)
plt.show()

[`jax.numpy` covers close to all np functions](https://docs.jax.dev/en/latest/jax.numpy.html).
JAX also covers [other NumPy submodules](https://docs.jax.dev/en/latest/jax.html) and even [SciPy ones](https://docs.jax.dev/en/latest/jax.scipy.html).
There are a however exceptions, for functions which have been deemed out of scope for JAX (numerical integration and minimization are examples of very limited support).

You now know how to use 80% of the JAX librarie.
The remaining 20% is actually the hard part and the subject of this whole tutorial/class.

## Arrays in JAX

JAX arrays mostly work like NumPy ones, with however a few important distinctions, covered in this part.

### Immutability

In JAX, all arrays are immutable. This means that once created, they can't be modified in-place.
Code such as this, while perfectly valid in NumPy, will fail:

In [ ]:
A = jnp.array([0., 1., 2., 10., 5., 4.])
A[1] = 2.

The correct thing to do, for individual elements (or slices), is to use [JAX indexed update syntax](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html#in-place-updates), which returnes an updated copy.
Many more operations are possible other than element assignation, such as addition or even the application of a function ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html), read it!).

In [ ]:
# Simply setting a scalar
A = A.at[1].set(2.)
print(A)

# Adding a scalar value to some element
A = A.at[2].add(3.)
print(A)

# Applying a function to some element
A = A.at[1].apply(lambda x: x**2)
print(A)

# Elements can be selected through slices too
A = A.at[2:4].set(2)
print(A)

The incurred copy will result in a loss of performance, compared to NumPy:

In [ ]:
def inplace_jax(A):
    return A.at[1].set(10.)

def inplace_numpy(A):
    A[1] = 10.
    return A

A_jax = jnp.ones(10_000)
A_np = np.ones(10_000)

%timeit inplace_jax(A_jax).block_until_ready()
%timeit inplace_numpy(A_np)

This important performance difference is completly expected.
First, from the array copy alone and also because JAX here runs in eager mode (non-jitted).
On the other hand, NumPy is very well optimised for singular function calls (with very well optimised implementations) and will mostly dominate over JAX perfomance wise on this kind of benchmark.

This will however change soon as we will introduce just-in-time compilation.

> _**NOTE:**_ To accurately time JAX calls, calling `.block_until_ready()` at the output array is necessary.
> This is because JAX [dispatches asynchronously](https://docs.jax.dev/en/latest/async_dispatch.html) and hands back an array before the computation actually finishes.
>On the other hand, NumPy is synchronous: the result is already there when the call returns.

### Out-of-bound indexing

In NumPy, it is expected that an error raises when indexing an array outside of its bounds:

In [ ]:
print(np.array([4., 2.])[4])

However, for technical reasons (difficulty of propagating errors on acceleration devices), [this error in JAX is silent](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html#out-of-bounds-indexing):

In [ ]:
print(jnp.array([4., 2.])[4])

It actually returns the last value of the array.
This behaviour can be changed by using the optionnal parameters of `.at[i].get()`:

In [ ]:
print(jnp.array([4., 2.]).at[1].get(mode='fill', fill_value=42.))
print(jnp.array([4., 2.]).at[4].get(mode='fill', fill_value=42.))

## Pure functions
Before we get to JIT compilation, one concept underlies everything that follows in JAX: [pure functions](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html#pure-functions).

A function is pure when both of these hold:

1. **Output depends only on the inputs.** Call it twice with the same arguments and you always get the same result.
2. **No side effects.** It changes nothing outside itself: no mutating globals, no I/O or printing no modifying its arguments in place.

Here are a few examples of pure and impure functions:

In [ ]:
# Examples of pure functions:

def f_pure(x):
    # Output is a function of x alone.
    return jnp.cos(x)

def f_pure2(x):
    # Also pure: the branch depends only on the input.
    if x >= 2.0:
        return jnp.cos(x)
    else:
        return jnp.sin(x)

def f_pure3(x):
    # Another pure function, this time using a for loop.
    y = 0
    for i in range(4):
        y = y + x[i]
    return y

def f_pure4(x, N):
    # Another pure function, this time using a for loop whose loop count is given as an input.
    y = 0
    for i in range(N):
        y = y + x[i]
    return y

def f_pure5(x):
    # Another pure function, this time using a for loop whose loop count is a function of the input values.
    y = 0
    for i in range(sum(x)):
        y = y + x[i]
    return y

def f_pureish(x):
    # Reading a global runs fine. But it isn't truly pure: the result depends
    # on external state, so changing c changes the output for the same x.
    # We will see that under jit c's value is frozen into the compiled function at trace time and won't
    # track later reassignments.
    return c + jnp.cos(x)


# Example of impure functions:

def f_impure(x):
    # print is I/O: a side effect. Under jit it fires once, during tracing, not on later calls.
    print(x)
    return jnp.cos(x)

c = 10.0

def f_impure2(x):
    # Rebinding a global is a side effect.
    global c
    c = x**2
    return jnp.cos(x)

def f_impure3(x):
    # Hidden global state: NumPy's RNG advances on every call, so the same x
    # yields different outputs. We will later see (on a separate notebook) how JAX handles this.
    return x + np.random.normal()

def f_impure4(x, history):
    # Mutating an argument in place is a side effect
    history.append(x)
    return jnp.cos(x)

## Just-in-time compilation (`jit`)

We now reach one of JAX's main feature: JIT compilation with `jax.jit`.

By default JAX runs eagerly, dispatching each operation to the device on its own.
That's flexible but slow: there's per-operation overhead and no room to optimize across operations.
On the other hand, `jax.jit` hands the compute function to [XLA](https://openxla.org/), JAX's compiler, which fuses the operations together and emits a single binary tuned for your target device (CPU, GPU, ...).

The first call to a jitted function does three things:

1. **Trace.** [JAX runs the Python body once, but with abstract stand-ins for the arguments](https://docs.jax.dev/en/latest/key-concepts.html#key-concepts-tracing): tracers that carry only shape and dtype without any concrete values. Instead of computing anything, it records the sequence of primitive operations into an intermediate form called a [jaxpr](https://docs.jax.dev/en/latest/key-concepts.html#jaxprs).
2. **Compile.** XLA turns that jaxpr into a binary, specialized to the shapes/dtypes it was traced with and to the device.
3. **Run.** The binary executes on the real inputs.

JAX then caches the binary.
A later call with arguments of the same shape and dtype skips straight to running.
However, changing a shape or dtype will retrigger JAX retracing and compilation, keeping a separate cached binary per signature.

This "trace once, run many times" model is why JAX demands pure functions.
Since the Python body runs only during that single trace, any side effect (a `print`, a global write, an RNG draw) happens once, at trace time, and never again on cached calls.

Two more consequences follow from tracers carrying shapes but not values:
- Anything depending on an input's shape works, since shapes are known while tracing.
- Anything depending on an input's value does not. A Python `if x > 0` or `for i in range(x)` over a traced value raises, because the value isn't available when Python needs to pick the branch or loop count.

JAX supplies structured replacements which we will see below and in subsequent notebooks.

As a first example, lets compile a silly function and compare its performance against eager mode and NumPy.

In [ ]:
def f_jax(x):
    return jnp.dot(jnp.cos(x)**2, jnp.sin(x)**2) + jnp.sin(jnp.exp(-x**2))/(jnp.cos(x+jnp.pi)+2.)

def f_np(x):
    return np.dot(np.cos(x)**2, np.sin(x)**2) + np.sin(np.exp(-x**2))/(np.cos(x+np.pi)+2.)

In [ ]:
A_jax = jnp.ones(100_000)
A_np = np.ones(100_000)

f_jax_jit = jax.jit(f_jax)
f_jax_jit(A_jax) # Run first to compile the function

%timeit f_np(A_np)
%timeit f_jax_jit(A_jax).block_until_ready()
%timeit f_jax(A_jax).block_until_ready()

### What is `jit` actually buying us here versus NumPy?

The benchmark computes the same quantity three ways:

**NumPy: excellent per call, blind across calls.**
Every NumPy ufunc (`cos`, `sin`, `^2`) is a hand-tuned, [SIMD](https://fr.wikipedia.org/wiki/Single_instruction_multiple_data)-vectorized C loop, and `dot` drops into [BLAS](https://fr.wikipedia.org/wiki/Basic_Linear_Algebra_Subprograms). Individually these are about as fast as CPU code gets.
But NumPy is eager and op-by-op: it never sees more than the single operation in front of it, so it evaluates the expression as a chain of full-array passes, each allocating a fresh temporary one.

**Jitted JAX: fusing.** `jit` traces the entire function into one computation graph and hands it to XLA, which fuses the elementwise chain and reduction into essentially a single compute kernel.
Per element: load $x_i$, compute the full operation chain and add to a running sum, all in registers.
As a result, no temporary array is needed.
On top of fusion, the whole-program view lets XLA do [constant folding](https://docs.jax.dev/en/latest/internals/constants.html) (and baking of global variables), [common-subexpression elimination](https://en.wikipedia.org/wiki/Common_subexpression_elimination), [algebraic simplification](https://openxla.org/xla/hlo_passes), and layout choices that NumPy can't attempt because it never has more than one op in view.
On top of that, JAX will be able to multithread elementwise computation while NumPy can't.

**Eager JAX: worst of both worlds.** Without `jit`, JAX runs op-by-op like NumPy (no fusion) and adds dispatch overhead through Python and its asynchronous runtime that NumPy's C path minimizes.
That overhead is roughly fixed per op.

> _**NOTE:**_ Eager mode should really be seen as a development/debugging convenience, or for performance non-critical operations such as when setting up a plot.
With JAX, the intended path is to always wrap computations with `jit`.

On top of calling `jax.jit` to produce a jitted function, a function can directly be jitted when decorated with the `@jax.jit` decorator:

In [ ]:
@jax.jit
def f(x):
    return jnp.cos(x)

f(jnp.ones(10))

Of course, jitted functions can be composed:

In [ ]:
@jax.jit
def g(x):
    return f(x)*jnp.exp(-x)

g(jnp.array([1.]))

Note that jit can be disabled using the `jax.disable_jit` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.disable_jit.html), read it!) context manager.
This can be useful to debug jitted functions which do not seem to work as expected (usually the results of some hidden side-effects)

In [ ]:
with jax.disable_jit():
    %timeit f_jax_jit(A_jax).block_until_ready()

%timeit f_jax_jit(A_jax).block_until_ready()

### NumPy can be faster than jitted JAX
NumPy, in some cases as it is a collection of very efficient compute routines, can be faster than JAX, especially when there is no operations to fuse.

This example calls the dot product, which in NumPy is backed by [OpenBLAS](https://github.com/OpenMathLib/OpenBLAS) (by default) and thus very fast.

> _**TODO:**_ Add/find XLA BLAS backend (is it OpenBLAS or inhouse?)

In [ ]:
def f_np(x):
    return np.dot(x, x)

@jax.jit
def f_jax(x):
    return jnp.dot(x, x)

f_jax(A_jax)

%timeit f_np(A_np)
%timeit f_jax(A_jax).block_until_ready()

### JIT and printing
As noted earlier, a `print` inside a function makes it impure: under `jit` the body runs only while tracing, so the print fires once at compile time and never on cached calls.
That limitation doubles as an useful tool: knowing exactly when a function is compiled and potentially recompiled.
Indeed, an unnoticed retrace inside a tight loop silently pays the full trace-and-compile cost on every iteration and will kill performance.

Inside a jitted function the argument is a tracer, and printing it shows its abstract value: shape, dtype, and the weak-type flag. [A value is weakly typed when its dtype was inferred from a Python scalar (`int`, `float`, `complex`) rather than pinned explicitly](https://docs.jax.dev/en/latest/type_promotion.html).
A weak type is shown with a leading tilde: `~int32[]` is weak, `int32[]` is strong.

The weak flag is part of the tracer's signature, so forcing a strong type retriggers compilation even when shape and dtype are unchanged: wrapping a value in `jnp.float32()` turns `~float32[]` into `float32[]`, a new signature.
Note the subtlety from before: it's the inferred dtype that makes a value weak, so `jnp.array(1.)` is still weak, whereas pinning the dtype (`jnp.array(1., dtype=jnp.float32)`) or building a non-scalar array yields a strong type.

Since `jit` keys on shape too, it distinguishes a scalar `jnp.array(1.)` (shape `()`)
from a size-1 array `jnp.array([1.])` (shape `(1,)`), which will retrigger a compilation.

> _**NOTE:**_ It is good practice, to avoid conversion between types, to always add a trailing dot after any scalar value meant to be float (like `1.`), even when the number is whole.

In [ ]:
@jax.jit
def f(x):
    print("Compiling f()... Type={}".format(x))
    return x

print(f(1))
print(f(5))
print(f(jnp.array(1)))
print(f(jnp.array(1, dtype=jnp.int32)))

print(f(4.))
print(f(jnp.float32(4.)))
print(f(jnp.array([4.])))

To print the value of a variable, even when jitted, `jax.debug.print` should be used ([Link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html), read it!).
It is somewhat less capable than `print` but does the job.

In [ ]:
@jax.jit
def g(x):
    jax.debug.print("Value={val}", val=x)
    return x

g(4.)
g(jnp.ones(2))

### JIT, pytrees, and static arguments

`jit` (and JAX's other transformations) don't only accept single arrays, they accept pytrees: arbitrarily nested containers of lists, tuples, and dicts with arrays (or scalars) at the leaves.
[Pytrees are a core JAX concept](https://docs.jax.dev/en/latest/key-concepts.html#pytrees).

Real problems often need richer structures as arguments to jitted functions, such as objects.
Custom classes are not pytrees by default: passing one to `jit` raises a `TypeError`, because JAX tries to read the object as an array.
They can, however, be registered as pytree nodes so transformations see through them to their array fields.
This will be explored in a later notebook.

On a call, `jit` flattens every argument into its array leaves and traces those
(shape/dtype tracers), while the pytree structure (container types, dict keys, tuple
lengths) becomes part of the compilation signature.
So changing a leaf's value (same shape) reuses the cache, whereas changing the structure (an extra dict key, a different tuple length) is a new signature and retraces.

When an argument genuinely must be concrete at trace time (like an integer that sets an array's shape, or a boolean flag), it needs to be marked as so using `static_argnums` (or `static_argnames`).
`jit` then uses the argument's value rather than a tracer, baking it into the compiled binary.

This has two consequences:
- The value must be hashable (ints, strings, bools, tuples of these are fine; lists,
  dicts, and JAX arrays are not), since it becomes part of the cache key. An unhashable
  static argument will raise an error.
- Changing that value is a cache miss, so `jit` retraces and compiles a fresh binary.

Some examples on how to flag an argument as static:

In [ ]:
# Using static_argnums which takes an argument position
@jax.jit(static_argnums=1)
def f_N(a, N):
    return jnp.full(a, N)

# Using static_argnums which takes an iterable, such as a tuple or list, of arguments positions
@jax.jit(static_argnums=(1, 2))
def f_NM(a, N, M):
    return jnp.full(a, (N, M))

# Flagging an argument as static using `static_argnames`
@jax.jit(static_argnames='flag')
def g_flag(flag):
    if flag:
        return jnp.zeros(10)
    else:
        return jnp.ones(10)

# As for `static_argnums`, `static_argnames` accept any iterable of strings
@jax.jit(static_argnames=('flag1', 'flag2'))
def g_2flags(flag1, flag2):
    if flag and flag2:
        return jnp.zeros(10)
    else:
        return jnp.ones(10)
            

The following will fail: JAX arrays are not hashable.

In [ ]:
@jax.jit(static_argnames='y')
def f(x, y):
    return jnp.dot(x, jnp.array(y))

x, y = jnp.ones(4), jnp.ones(4)
f(x, y)

The following will fail: classes are not pytrees by default.

In [ ]:
class A:
    def __init__(self, v):
        self.val = v

@jax.jit
def f(x, obj):
    return x*obj.val

f(2., A(4.))

Passing the object as a static argument will work. However, as it is baked in the compilation, changing its field will retrigger a new compilation.
We will see later on how to avoid this.

In [ ]:
@jax.jit(static_argnames='obj')
def f(x, obj):
    print("Compiling with val={}".format(obj.val))
    return x*obj.val

a = A(4.)
f(2., A(4.))
f(2., A(5.))

## Python loops under `jit`

Python `for` and `while` loops are fine inside a jitted function as long as the number of iterations is fixed at trace time.
Because tracing executes the Python body, it walks straight through such a loop, running every iteration and recording its operations.
The loop control never survives into the compiled program: the loop is unrolled, its body stamped out once per iteration, with the loop index appearing as a concrete constant each time.
Depending on the situation, this behaviour can be fine but will incure long compilation time if the loop count is high or the iteration computation is complex (I've personnaly seen compilation time exceeding 30 min because of this).

We will see later on how minimize compilation time and have dynamic loop count in a separate notebooks.

In [ ]:
@jax.jit(static_argnames='N')
def f(x, N):
    # Loop count is encoded as a static parameter.
    y = 0.
    for i in range(N):
        y += x
    return y

f(10., 5)

And, as expected, if the loop count is only known at run time, it will fail.

In [ ]:
@jax.jit
def f(x):
    N = int(jnp.sum(x)) # This value is only known at runtime
    y = 0.
    for i in range(N):
        y += x
    return y

f(jnp.ones(4))

## JIT and slices
As we saw before, slices of array can be retrieved and set using the `.at[]` interface.
This works fine with JIT if the slice is defined as static or is constant:

In [ ]:
@jax.jit
def f(x):
    return x.at[2:5].get()

@jax.jit(static_argnames=('N', 'M'))
def g(x, N, M):
    return x.at[N:M].get()

A = jnp.arange(10)
print(f(A))
print(g(A, 2, 5))

However, if one (or both) of the slice boundary is dynamic, it will fail as expected:

In [ ]:
@jax.jit
def f(x, y):
    N = jnp.min(y)
    M = jnp.max(y)
    return x.at[N:M].get()

f(A, jnp.array([4., 2., 6.]))

There are several ways to get around this.

If for example the length of the slice is known at compile time, but the start of it is not, then `jax.lax.dynamic_slice` ([link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.dynamic_slice.html), read it!) can be used:

In [ ]:
@jax.jit(static_argnames='length')
def f(x, y, length):
    start = jnp.min(y)
    # Note that the "start_indices", "slice_sizes" arguments needs to be iterables,
    # with size matching the dimension of the operand array (see the documention)
    return jax.lax.dynamic_slice(x, (start,), (length,))

f(A, jnp.array([4, 2, 6]), 4)

Getting around this limitation is one of the main difficulty when dealing with jitted JAX and will sometime require creative solutions (which we will explore in some exercices).

## Basic control flow

As we've seen a few times now, tracing runs your function on tracers: placeholders that carry an array's shape and dtype but no concrete value.
Computations on those values are exactly what tracing records.
What breaks is Python control flow that has to branch on a value.

An `if` or `while` needs a concrete `True`/`False` to choose a path and a tracer holds no value to give them.
So Python's attempt to convert the tracer to a `bool` (or `int`) fails, and the trace raises instead of compiling.

The important consequence is that purity alone doesn't make a function jit-able.
A function can depend solely on its inputs and cause no side effects, yet still be rejected by `jit` because it branches on a traced value.
The function below is perfectly pure (and runs correctly in eager mode) but does not compile:

In [ ]:
@jax.jit
def f(x):
    if jnp.sum(x) > jnp.pi:
        return x**2
    else:
        return jnp.cos(x)

f(4.)

[JAX provides control flow operations compatible with JIT transformations.](https://docs.jax.dev/en/latest/control-flow.html)
In this case, the solution is to use `jax.lax.select`([Link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.select.html), read it!).

>_**NOTE:**_ The `jax.lax` submodule is JAX's low-level layer of primitive operations,
>most of them thin wrappers around XLA.
>`jax.numpy` is built on top of it: `jnp` is the familiar, high-level interface, while `lax` is stricter and lower-level.
>It's also home to the structured control-flow primitives we explore now.

In [ ]:
@jax.jit
def f(x):
    return jax.lax.select(x > jnp.pi, x**2, jnp.cos(x))

# Work as intended
print(f(4.))

# Works on array, jax.lax.select is broadcastable
print(f(jnp.array([jnp.pi/2., 5., 6.])))

# Same, but this time providing a mask array
@jax.jit
def g(x, mask):
    return jax.lax.select(mask, x**2, jnp.cos(x))

mask = jnp.array([True, False, True, True])
A = jnp.array([1., 2., 3., 4.])
g(A, mask)

To select a branch depending on a condition, use `jax.lax.cond`.
This is like using a `if ... else ...` control flow on a scalar boolean, but without the recompilation issue.

In [ ]:
@jax.jit
def f(x, flag): # `flag` is not set as a static argument
    print("Compiling... Tracer={}".format(x))
    def _on_true(x):
        return x**2

    def _on_false(x):
        return jnp.cos(x)

    return jax.lax.cond(flag, _on_true, _on_false, x)

# On scalar input
print(f(4., True))
print(f(4., False))


# On array input
f(jnp.array([jnp.pi/2., 5., 6.]), True)
f(jnp.array([jnp.pi/2., 5., 6.]), False)

[There are many more control flow operators in JAX.](https://docs.jax.dev/en/latest/control-flow.html#structured-control-flow-primitives)

This includes:
- `jax.numpy.where`: [mirrors `np.where`](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.where.html),
- `jax.lax.switch`: [like `lax.cond`, but allows switching between any number of callable choices](https://docs.jax.dev/en/latest/_autosummary/jax.lax.switch.html),
- `jax.numpy.piecewise`: [a numpy-style wrapper of `lax.switch`, but switches on a list of boolean conditions rather than a single scalar index](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.piecewise.html).

Other control flow operators exists for loops and will be explored in dedicated notebooks.

Now that you are somewhat familiar with JAX and its JIT framework, you should continue with the Vmap notebook which will teach you one other core JAX transformation: `jax.vmap`.